###books recommender system using clustering| collaborative based 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
books = pd.read_csv('data/BX-Books.csv',sep=";", on_bad_lines='skip', encoding='latin-1')

C:\Users\Shrey\AppData\Local\Temp\ipykernel_19888\1149443711.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('data/BX-Books.csv',sep=";", on_bad_lines='skip', encoding='latin-1')


In [3]:
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')

In [4]:
books=books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-L']]

In [5]:
books.rename(columns={
    "Book-Title":"title",
    "Book-Author":"author",
    "Year-Of-Publication":"year",
    "Publisher":"publisher",
    "Image-URL-L":"img_url"},inplace = True)

In [6]:
users=pd.read_csv('data/BX-Users.csv',sep=";", on_bad_lines='skip', encoding='latin-1')

In [7]:
ratings=pd.read_csv('data/BX-Book-Ratings.csv',sep=";", on_bad_lines='skip', encoding='latin-1')

In [8]:
print(books.shape)
print(users.shape)
print(ratings.shape)

(271360, 6)
(278858, 3)
(1149780, 3)


In [9]:
ratings.rename(columns={
    "User-ID":"user_id",
    "Book-Rating":"rating"}, inplace = True)

In [10]:
x = ratings['user_id'].value_counts()>200

In [11]:
y = x[x].index

In [12]:
ratings = ratings[ratings['user_id'].isin(y)]

In [13]:
ratings_with_books = ratings.merge(books,on = "ISBN")

In [14]:
num_rating =ratings_with_books.groupby("title")["rating"].count().reset_index()

In [15]:
num_rating.rename(columns={
    "rating": "num_of_rating"}, inplace= True)

In [16]:
final_rating = ratings_with_books.merge(num_rating, on="title")

In [17]:
final_rating = final_rating[final_rating['num_of_rating']>=50]

In [18]:
final_rating.drop_duplicates(["user_id","title"],inplace = True)

In [19]:
book_pivot = final_rating.pivot_table(columns='user_id', index='title', values='rating')

In [20]:
book_pivot.fillna(0,inplace= True)

In [21]:
from scipy.sparse import csr_matrix

In [22]:
book_sparse = csr_matrix(book_pivot)

In [23]:
from sklearn.neighbors import NearestNeighbors
model = NearestNeighbors(algorithm = 'brute')

In [24]:
model.fit(book_sparse)

NearestNeighbors(algorithm='brute')

In [25]:
books_name = book_pivot.index

In [26]:
import joblib

joblib.dump(model, 'artifacts/model.pkl')
joblib.dump(books_name, 'artifacts/books_name.pkl')
joblib.dump(final_rating, 'artifacts/final_rating.pkl')
joblib.dump(book_pivot, 'artifacts/book_pivot.pkl')

['artifacts/book_pivot.pkl']

In [27]:
def recommend_book(book_name):
    book_id = np.where(book_pivot.index == book_name)[0][0]
    distance,suggestion = model.kneighbors(book_pivot.iloc[book_id,:].values.reshape(1,-1),n_neighbors=6)

    for i in range(len(suggestion)):
        books= book_pivot.index[suggestion[i]]
        for j in books:
            print(j)

In [28]:
book_name='You Belong To Me'
recommend_book(book_name)

You Belong To Me
The Cradle Will Fall
Exclusive
Loves Music, Loves to Dance
While My Pretty One Sleeps
Before I Say Good-Bye


In [29]:
import joblib
books_name = joblib.load('artifacts/books_name.pkl')
print(type(books_name))

<class 'pandas.core.indexes.base.Index'>


In [30]:
!where python

C:\Users\Shrey\anaconda3\envs\bookrec\python.exe
C:\Program Files\Python38\python.exe
